# 1.3 · Filing text: annual reports and the filing index

> **Needs:** notebook 1.1 (its companies with an SEC number); for FinBERT the optional `text` and `deep` dependency groups; about 10 GB of disk.
>
> **Produces:** the filing index and the text measures (`equity.document`, `equity.text_measure`) that notebook 5.10 reads - no other notebook needs them.

Notebooks 1.1 and 1.2 load numbers: prices, accounts, estimates. This one loads **words**. Every US public company
files an annual report (the **10-K**) with the SEC, and a **current report** (the **8-K**) whenever something material
happens - results, an executive leaving, an acquisition. Notebook 5.10 asks whether that text helps to rank stocks;
this notebook loads what it reads:

| Data | Table | Contents |
|---|---|---|
| **Filing index** | `equity.document` | every 10-K, 10-Q and 8-K each company filed since 2000 (amendments too): form, filing date, the instant the SEC accepted it, and an 8-K's item codes (what happened) |
| **Annual reports** | raw zone, `sec/annual_report/` | the main document of every original 10-K, as the SEC serves it, compressed - about 47,500 documents and 10 GB for the S&P 1500 since 2003 (more since 2000) |
| **Text measures** | `equity.text_measure` | per report, for the whole text, the management's discussion (MD&A) and the risk factors: word counts by category of the Loughran-McDonald dictionary, the similarity to the company's previous report, and FinBERT's tone of the MD&A |

* **Run after notebook 1.1.** Companies are found by their SEC number (the **CIK**), which the SEC step of the
  initial load attaches. A company without a CIK has no filings here.
* **Run time:** about 2 hours for the downloads (the SEC allows 10 requests a second; four threads share that limit),
  an hour to measure the text, and about 75 minutes for FinBERT on a GPU. **Without a GPU, FinBERT takes from half
  a day to two days** - set `TONE = False` below and everything else still loads.
* **Idempotent and resumable.** A document already landed is never fetched again; a company whose reports are all
  measured is not measured again; FinBERT's scores wait in a checkpoint file while it runs. A run that stops - a lost
  connection, a closed laptop - is simply run again and continues.
* **Prerequisites:** a contact name for the SEC's API (`EQR_SEC_USER_AGENT`, as for notebook 1.1); for FinBERT the
  dependency group `text` (`uv sync --all-groups`). No API key and no cost: the dictionary and FinBERT are free.

## What is measured, and why

* **Loughran-McDonald dictionary.** Word lists built for financial documents: negative, positive, uncertain,
  litigious, constraining, and strong and weak modal words ("must", "may"). General-purpose sentiment lists call
  "liability" or "tax" negative; in a 10-K they are ordinary. Each category's count is stored for the whole report,
  the MD&A (Item 7) and the risk factors (Item 1A).
* **Similarity to last year's report.** Most companies copy last year's 10-K with small edits. Two measures of how
  much changed: **cosine similarity** (do the two reports use words in the same proportions?) and **Jaccard
  similarity** (how much of their vocabulary is shared?). Cohen, Malloy and Nguyen ("Lazy Prices", 2020) found that
  companies whose reports change most did worse afterwards.
* **FinBERT.** A language model (BERT, trained further on financial text) that reads a sentence and says whether it
  is positive, negative or neutral. It scores 32 sentences spread through each MD&A; the report's tone is their
  average. It is a second opinion on the dictionary: a model reads "losses narrowed" as good news, a word count
  does not.

Two design rules from the architecture apply here as everywhere:

* **Never fetch twice.** Each document is stored in the raw zone exactly as the SEC sent it. A better way to find the
  MD&A, or a new dictionary, is applied to the stored documents; nothing is downloaded again.
* **Two clocks.** Each filing carries the instant the SEC accepted it (`published_at`). A backtest deciding on a
  month end reads only reports accepted *before* it - using a report on the day of its period end, months before it
  was written, would be look-ahead bias.

In [1]:
import logging

import pandas as pd
import sqlalchemy as sa
from IPython.display import display

from equity_research.research.inventory import table_rows
from equity_research.research.notebook import shown
from research_core.data.db.engine import get_engine
from research_core.settings import get_settings

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# progress lines while the load runs (every 250 indexes, 500 documents, 100 companies, 200 FinBERT reports)
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S", force=True)
for noisy in ("httpx", "httpcore", "urllib3", "alembic", "filelock", "huggingface_hub"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

SINCE = "2000-01-01"  # filings from the day the data begins (a report's similarity needs the one before: from 2001)
TONE = True  # FinBERT's tone of each MD&A; False loads everything else

settings = get_settings()
engine = get_engine()
print("Database:     ", settings.database.server, "/", settings.database.name)
print("Files kept in:", shown(settings.raw_dir))
with engine.connect() as conn:
    companies = conn.execute(
        sa.text(
            "SELECT COUNT(DISTINCT entity_id) FROM core.identifier WHERE entity_type = 'issuer' AND id_type = 'CIK'"
        )
    ).scalar()
print(f"{companies:,} companies have an SEC number (CIK) - run notebook 1.1 first if this is 0")

Database:      MENONPC\SQLEXPRESS / quant_research_local
Files kept in: data/quant_research_local/raw
3,293 companies have an SEC number (CIK) - run notebook 1.1 first if this is 0


## Step 1 · Where FinBERT runs

FinBERT is a neural network: on a graphics card (GPU) it scores a report in under a tenth of a second; on this
PC's processor in about a second when nothing else is running, and two to four seconds beside other work - with
47,500 reports, the difference between an hour and a day or two. `best_device()` picks the GPU when torch can use
one. A GPU's scores differ from a processor's only in the seventh decimal place.

In [2]:
DEVICE = "cpu"
if TONE:
    from equity_research.ml.deep import best_device

    DEVICE = best_device()
    if DEVICE == "cuda":
        import torch

        print("FinBERT will run on the GPU:", torch.cuda.get_device_name(0))
    else:
        print("No GPU found: FinBERT will run on the processor - half a day or more. Set TONE = False to skip it.")
else:
    print("FinBERT is switched off (TONE = False)")

FinBERT will run on the GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Step 2 · Load

One call runs every step; each is a job of its own in `ops.job_run`:

1. **Indexes** - each company's list of filings from the SEC (`data.sec.gov/submissions`), with the older pages a
   long-lived company's list is split into. Fetched again only when our copy is over a month old.
2. **Documents** - the filing index written to `equity.document`.
3. **Annual reports** - the main document of every original 10-K (also the older form 10-K405 and the transition
   report 10-KT) not yet landed. A report two companies file together - a parent and its subsidiary - is one
   document: it is fetched once and measured for both.
4. **Dictionary** - the Loughran-McDonald word lists, once.
5. **Measures** - word counts and similarities, companies in parallel on every processor core but one.
6. **Tone** - FinBERT, when `TONE` is on.

The downloads and FinBERT's scoring do no database work: files wait in a staging folder and the scores in a
checkpoint, and each is written in a short burst. That matters on Azure's free serverless database, whose monthly
allowance counts every second it is awake (run this early in the month there: the first full load, with its
restarts, used up what was left of November 2026's).

In [3]:
from equity_research.pipelines.load_sec_text import load_sec_text

report = load_sec_text(engine, since=SINCE, tone=TONE, tone_device=DEVICE)
print(f"{report['ciks']:,} company SEC numbers asked; {report['ciks_unknown_to_edgar']} unknown to the SEC")
display(pd.DataFrame({k: report[k] for k in ("documents", "annual_reports", "measures", "tone") if k in report}).T)

21:39:25  SEC indexes: 250 / 3304 CIKs


21:41:43  SEC indexes: 500 / 3304 CIKs


21:43:59  SEC indexes: 750 / 3304 CIKs


21:45:56  SEC indexes: 1000 / 3304 CIKs


21:47:40  SEC indexes: 1250 / 3304 CIKs


21:49:29  SEC indexes: 1500 / 3304 CIKs


... (564 outputs left out)

07:52:53  FinBERT: 52200 / 52546 reports


07:54:00  FinBERT: 52400 / 52546 reports


3,304 company SEC numbers asked; 0 unknown to the SEC


,inserted,updated,landed,already,missing
documents,899544.0,0.0,NaN,NaN,NaN
annual_reports,NaN,NaN,52474.0,0.0,798.0
measures,1360290.0,0.0,NaN,NaN,NaN
tone,201472.0,0.0,NaN,NaN,NaN


In the table: `documents` are rows of the filing index, `measures` and `tone` rows of `equity.text_measure` (each
*inserted*, *updated* or left *unchanged*); `annual_reports` counts documents - *landed* now, *already* held, and
*missing* at the SEC (a filing whose main document it no longer serves).

## Step 3 · Checks

The quality gate asks whether any landed report has no measures - a load that stopped between landing and measuring.
It should read `pass` with a value of 0. Then the filing index by form, and how many annual reports have each kind
of measure.

In [4]:
from equity_research.quality.checks import CORE_CHECKS
from research_core.data.quality import run_checks

text_checks = [c for c in CORE_CHECKS if c.name in ("annual_reports_without_text_measures", "rows_without_provenance")]
results = run_checks(engine, text_checks)
display(pd.DataFrame([{"check": r.name, "status": r.status, "value": r.value, "levels": r.threshold} for r in results]))

with engine.connect() as conn:
    forms = pd.read_sql(sa.text("SELECT form, COUNT(*) AS filings FROM equity.document GROUP BY form"), conn)
    kinds = pd.read_sql(
        sa.text(
            "SELECT method, section, COUNT(DISTINCT filing_ref) AS reports, COUNT(DISTINCT issuer_id) AS companies "
            "FROM equity.text_measure GROUP BY method, section ORDER BY method, section"
        ),
        conn,
    )
display(forms.sort_values("filings", ascending=False).set_index("form").T)
display(kinds)

,check,status,value,levels
0,rows_without_provenance,pass,0.0,"warn <= 0, fail <= 1000000000.0"
1,annual_reports_without_text_measures,pass,0.0,"warn <= 0.01, fail <= 0.1"


form,8-K,10-Q,10-K,8-K/A,10-K/A,10-Q/A,10-K405,10-K405/A,10-KT,10-KT/A
filings,649687,164740,52750,17333,7087,5613,1980,248,92,14


,method,section,reports,companies
0,bow-1,all,49067,3170
1,bow-1,mdna,42703,3060
2,bow-1,risk,35373,2868
3,finbert-prosus,mdna,50292,3207
4,lm-1993-2025,all,52470,3226
5,lm-1993-2025,mdna,46559,3135
6,lm-1993-2025,risk,38962,2972


Reading the second table: `lm-1993-2025` is the dictionary (every report has the section `all`; `mdna` and `risk`
only where the 10-K contains them - some companies put the MD&A in an exhibit, and risk factors became compulsory
with the 2005 annual reports), `bow-1` the similarity to the previous report (absent for a company's first), and
`finbert-prosus` FinBERT's tone (reports whose MD&A has prose sentences).

How many reports each year, and how many of them have an MD&A and risk factors that could be found:

In [5]:
with engine.connect() as conn:
    yearly = pd.read_sql(
        sa.text(
            "SELECT YEAR(d.filing_date) AS year, COUNT(DISTINCT d.filing_ref) AS reports, "
            "COUNT(DISTINCT CASE WHEN m.section = 'mdna' THEN m.filing_ref END) AS with_mdna, "
            "COUNT(DISTINCT CASE WHEN m.section = 'risk' THEN m.filing_ref END) AS with_risk_factors, "
            "COUNT(DISTINCT CASE WHEN m.method = 'finbert-prosus' THEN m.filing_ref END) AS with_finbert "
            "FROM equity.document d LEFT JOIN equity.text_measure m "
            "ON m.issuer_id = d.issuer_id AND m.vendor_id = d.vendor_id AND m.filing_ref = d.filing_ref "
            "WHERE d.form IN ('10-K', '10-K405', '10-KT') GROUP BY YEAR(d.filing_date) ORDER BY 1"
        ),
        conn,
    )
display(yearly.set_index("year").T)

year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
reports,2044,2084,2086,2106,2112,2137,2140,2143,2136,2108,2112,2106,2084,2072,2082,2086,2053,2016,1993,1979,1945,1929,1914,1893,1853,1832,1701
with_mdna,317,1447,2004,2033,2043,2072,2078,2076,2071,2035,2022,2035,2019,2016,2019,1987,1936,1899,1877,1868,1859,1853,1831,1805,1763,1733,1594
with_risk_factors,1,2,5,4,5,51,1957,2016,2029,1991,2000,1995,1972,1989,2004,1961,1879,1824,1812,1798,1803,1806,1725,1646,1610,1592,1485
with_finbert,317,1447,2004,2033,2043,2072,2078,2076,2071,2035,2022,2035,2019,2016,2019,1987,1936,1899,1877,1868,1859,1853,1831,1805,1763,1733,1594


## Step 4 · Sync the research cache

Research reads the local copy of the database. The text tables have a **data version of their own**: loading them
does not make every earlier price- and accounts-based result stale; notebook 5.10, which reads both, keys its results
by both versions.

In [6]:
from equity_research.data import artifacts as store
from equity_research.data.cache import ResearchCache

cache = ResearchCache()
copied = cache.sync(engine)
display(pd.Series({k: v for k, v in copied.items() if v}, name="rows copied"))
print("prices and accounts version:", store.data_snapshot(cache))
print("text version:               ", store.data_snapshot(cache, tables=sorted(store.SEPARATE_TABLES)))
rows = table_rows(engine)
display(rows[rows["table"].isin(["document", "text_measure", "batch"])])

07:56:04  cache core.vendor: 11 rows


07:56:04  cache core.issuer: 5177 rows


07:56:04  cache equity.classification: 7162 rows


07:56:04  cache core.listing: 5332 rows


07:56:05  cache core.identifier: 33157 rows


07:56:06  cache equity.index_definition: 6 rows


... (14 outputs left out)

07:57:51  cache research.strategy_rule: 234 rows


core.vendor                       11
core.issuer                     5177
equity.classification           7162
core.listing                    5332
core.identifier                33157
equity.index_definition            6
equity.index_membership         8635
equity.universe                    8
equity.universe_component         12
equity.document               899544
equity.text_measure          1561762
research.rule_set                  1
research.strategy_rule           234
Name: rows copied, dtype: int64

prices and accounts version: 2026-11-16T21:22:58
text version:                2026-11-17T07:55:28


,schema,table,rows
9,equity,document,899544
19,equity,text_measure,1561762
27,ingest,batch,62670


## Running it again

A second run fetches and measures nothing; run monthly (or after notebook 1.2) it picks up the new filings: an index
is fetched again when our copy is over a month old (`max_age_days`), new 10-Ks are landed and measured, and a company
with a new report is measured again in full (its similarity needs the report before).

The same load from the command line:

```
uv run eqr load-sec-text --verbose --tone --tone-device cuda
```

`--issuers 12,34` limits a trial to some companies; without `--tone` FinBERT is skipped.

## Glossary

* **10-K / 10-Q / 8-K** - a US company's annual report / quarterly report / report of a material event, filed with
  the SEC. **10-K405** is an older form of the 10-K, **10-KT** one covering a changed financial year; **/A** marks an
  amendment.
* **EDGAR** - the SEC's public filing system. **CIK** - the number it gives each filer. **Accession number** - the
  number of one filing (`0000320193-24-000123`); a report filed jointly has one accession number under two CIKs.
* **MD&A** - Item 7 of the 10-K: management's discussion and analysis of the year.
* **Risk factors** - Item 1A: the risks management lists; compulsory from the 2005 annual reports.
* **Loughran-McDonald dictionary** - word lists of negative, positive, uncertain, litigious, constraining and modal
  words, built from 10-Ks (Loughran and McDonald, 2011; the 1993-2025 edition is used).
* **Cosine similarity** - 1 when two documents use their words in the same proportions, lower as they differ.
  **Jaccard similarity** - the share of the two documents' vocabularies that they have in common.
* **FinBERT** - a BERT language model fine-tuned to classify financial sentences (ProsusAI's, from Hugging Face).
* **GPU** - a graphics card; neural networks run many times faster on one than on a processor.
* **Raw zone / look-ahead bias / point-in-time** - as in notebook 1.1.

**Next:** notebook 5.10 turns these measures into features and tests them.